# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIRˆ² dataset using the [`mlcroissant`](https://mlcroissant.org) library. 

The dataset contains ordered logistic regression outputs for household adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and explore basic information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The Dataset.metadata attribute exposes metadata fields as attributes
metadata = dataset.metadata

print(f"\033[1mDataset Name:\033[0m {metadata.name}")
print(f"\033[1mDescription:\033[0m {metadata.description}")
print(f"\033[1mDate Published:\033[0m {metadata.datePublished}")
print(f"\033[1mVersion:\033[0m {metadata.version}")
print(f"\033[1mIdentifier (DOI):\033[0m {metadata.identifier}")
print(f"\033[1mLicense:\033[0m {metadata.license}")

## 2. Data Overview

List available record sets, their IDs (`@id`), and their fields (with `@id` references). This is essential for identifying which data to extract.

**Note:** All entities are referenced by their `@id`.

In [ ]:
# List available record sets in the dataset
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    print(f"Record sets found ({len(record_sets)}):")
    for rs in record_sets:
        print(f"- Name: {rs.name} | @id: {rs.id_}")
        if rs.fields:
            print("  Fields:")
            for fld in rs.fields:
                print(f"    - Name: {fld.name} | @id: {fld.id_}")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - Name: {col.name} | @id: {col.id_}")
        print()

## 3. Data Extraction

Extract data from one or more record sets using their `@id`.

Below, we gather all record set IDs and load each corresponding record set’s data into a DataFrame. 
All entities are referenced by their `@id`.

In [ ]:
# Step 1: Get all record set IDs
# This block scans for all record sets and uses their id_ (the @id)
record_sets = list(dataset.record_sets())
record_set_ids = [rs.id_ for rs in record_sets]
print("Record set @ids detected:")
for rsid in record_set_ids:
    print(f"  - {rsid}")

# Step 2: Load data into DataFrames
dataframes = dict()
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"\nLoaded data for record set '@id': {rsid}")
        print(f"Shape: {df.shape}")
        print(f"Columns (@id): {list(df.columns)}")
    except Exception as ex:
        print(f"Warning: Could not load record set {rsid} due to: {ex}")

# For demonstration, select the first available record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nFirst record set chosen for EDA: {chosen_record_set_id}")
    first_df = dataframes[chosen_record_set_id]
    print(f"First 5 rows:\n{first_df.head()}")
else:
    print("No record sets with extractable tabular data were found.")

## 4. Exploratory Data Analysis (EDA)

Demonstrate a few typical preprocessing and EDA techniques:

- Filter records by a numeric criterion.
- Normalize a numeric field.
- Group by a categorical field for summarization.

**All fields and columns are referenced by their `@id`.**

In [ ]:
# The following is a generic EDA template.
# Please adapt 'numeric_field_id' and 'group_field_id' to match your dataset's actual @id values from the previous overview.

if record_set_ids:
    df = dataframes[chosen_record_set_id]
    # Attempt to identify numeric and group fields by inspecting columns and data types
    print(f"\nColumns in record set {chosen_record_set_id}:")
    for col in df.columns:
        print(f"  - {col}: dtype={df[col].dtype}")

    # Heuristic: pick the first numeric (float/int) column and one likely group/category column
    numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    non_numeric_cols = [col for col in df.columns if not np.issubdtype(df[col].dtype, np.number)]

    if not numeric_cols:
        print("No numeric fields found in the DataFrame for numeric EDA.")
    else:
        # Use the first numeric field's @id
        numeric_field_id = numeric_cols[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")

        # Choose first non-numeric as a group field (if possible)
        group_field_id = non_numeric_cols[0] if non_numeric_cols else None

        # Choose a dynamic threshold - mean if plausible or 10 otherwise
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()

        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the selected categorical/grouping field
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical/group field found for grouping analysis.")
else:
    print("Skipping EDA: no record sets loaded.")

## 5. Visualization

Visualize data distributions and relationships. The following examples use matplotlib and seaborn for illustrative purposes—substitute with specific field `@id`s from the previous steps as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use same selected record set and numeric field as above
if record_set_ids and numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion

- Demonstrated loading, overview, extraction, and EDA on the FAIR^2 dataset using `mlcroissant`.
- All entities (record sets, fields, columns) were referenced by their `@id`, as recommended by the Croissant metadata standard.
- This workflow can be adapted for any Croissant-structured FAIR dataset.

Further steps may include building predictive models, in-depth cleaning, or cross-record set joining depending on your analysis objectives.

---
*Notebook template inspired by the [mlcroissant](https://mlcroissant.org) documentation and consistent with the FAIR data principles.*